# Hybrid QLSTM for Library Occupancy Forecasting: full code
Reproduces every result in the report: simulated dataset, QLSTM (PennyLane VQC gates), LSTM / ARIMA / naive baselines, ablations, tables and figures.

**Colab:** Runtime → Run all. CPU is enough (no GPU needed).
Full run is about 45 to 60 min on Colab CPU (QLSTM simulation is the slow part). Set `QUICK = True` in the first code cell for a ~5 min smoke test (numbers will differ from the report).

In [ ]:
!pip -q install pennylane statsmodels

In [ ]:
QUICK = False          # True = tiny smoke test; False = full experiment reported in the report
import json, time, copy, numpy as np, pandas as pd, torch, torch.nn as nn, pennylane as qml
import matplotlib.pyplot as plt
torch.set_num_threads(2)
CFG = dict(seeds=[0] if QUICK else [0,1,2], q_epochs=3 if QUICK else 25, l_epochs=10 if QUICK else 80,
           abl_epochs=2 if QUICK else 15, nocal_epochs=2 if QUICK else 25)
print(CFG, 'pennylane', qml.__version__, 'torch', torch.__version__)

## 1. Simulated occupancy dataset
No real logs were available, so the series is generated from timetable and exam patterns (bimodal daily profile, weekday/weekend scaling, revision-week ramp, exam-week surge with a late-evening peak, holidays, AR(1) noise). Exam and holiday effects are built in, which is stated in the report.

In [ ]:
CAP=100.0; OPEN=list(range(8,22)); H=6; L=8      # capacity %, open hours 08-21, horizon 6 h, window 8 h
def make_data(seed=7, weeks=16):
    rng=np.random.default_rng(seed); rows=[]
    start=pd.Timestamp('2026-08-03')              # a Monday
    exam_weeks={6,14,15}                           # mid-sem week 7, end-sem weeks 15-16
    holidays={19,44,52,95}                         # weekday holidays (95 falls in the test set)
    noise=0.0
    for d in range(weeks*7):
        date=start+pd.Timedelta(days=d); dow=date.dayofweek; wk=d//7
        exam=int(wk in exam_weeks); hol=int(d in holidays); pre=int((wk+1) in exam_weeks)
        dscale={0:1,1:1.05,2:1.0,3:1.0,4:0.85,5:0.6,6:0.35}[dow]
        if hol: dscale*=0.2
        scale=dscale*(1+0.6*exam+0.2*pre)*(1+0.1*np.sin(wk/3))
        for h in OPEN:
            prof=0.55*np.exp(-((h-11)/2.3)**2)+0.75*np.exp(-((h-16)/2.6)**2)+0.12
            if exam: prof+=0.45*np.exp(-((h-20)/2.0)**2)
            noise=0.7*noise+rng.normal(0,0.05)
            occ=CAP*0.62*prof*scale*(1+noise)+rng.normal(0,1.5)
            rows.append((date+pd.Timedelta(hours=h),float(np.clip(occ,3,CAP)),h,dow,exam,hol))
    return pd.DataFrame(rows,columns=['ts','occ','hour','dow','exam','holiday'])
def features(df):
    h=df.hour.values
    return np.stack([df.occ.values/CAP,np.sin(2*np.pi*(h-8)/14),np.cos(2*np.pi*(h-8)/14),
                     df.dow.values/6.0,df.exam.values,df.holiday.values],1).astype(np.float32)
def windows(df):
    X=features(df); y=df.occ.values/CAP; n=len(df); idx=np.arange(L-1,n-H)
    Xw=np.stack([X[i-L+1:i+1] for i in idx]); Y=np.stack([y[i+1:i+1+H] for i in idx]).astype(np.float32)
    return Xw.astype(np.float32),Y,idx
def split(idx):
    n=idx.max()+H+1; a=int(0.7*n); b=int(0.8*n)
    return idx+H<a, (idx>=a-H)&(idx+H<b), idx>=b-H     # chronological train / val / test
df=make_data(); Xw,Y,idx=windows(df); TR,VA,TE=split(idx)
print(len(df),'observations | windows train/val/test =',TR.sum(),VA.sum(),TE.sum())
print(df.occ.describe().round(2))
df.plot(x='ts',y='occ',figsize=(10,2.5),legend=False,lw=0.6,ylabel='Occupancy (%)'); plt.show()

## 2. Models: QLSTM (VQC gates) and classical LSTM

In [ ]:
class VQC(nn.Module):
    """Angle encoding (H, RY(v), RZ(v^2)) -> [CNOT ring + trainable RY,RZ] x layers -> <Z> on every qubit."""
    def __init__(s,n,layers):
        super().__init__(); s.n=n; dev=qml.device('default.qubit',wires=n)
        @qml.qnode(dev,interface='torch',diff_method='backprop')
        def circ(inp,w):
            for i in range(n): qml.Hadamard(i); qml.RY(inp[...,i],wires=i); qml.RZ(inp[...,i]**2,wires=i)
            for l in range(layers):
                for i in range(n): qml.CNOT(wires=[i,(i+1)%n])
                for i in range(n): qml.RY(w[l,i,0],wires=i); qml.RZ(w[l,i,1],wires=i)
            return [qml.expval(qml.PauliZ(i)) for i in range(n)]
        s.circ=circ; s.w=nn.Parameter(0.1*torch.randn(layers,n,2))
    def forward(s,x): return torch.stack(s.circ(x,s.w),-1).float()
class QLSTM(nn.Module):
    def __init__(s,nf=6,n=4,layers=2,H=6):
        super().__init__(); s.n=n; s.inp=nn.Linear(nf+n,n)
        s.vqc=nn.ModuleList([VQC(n,layers) for _ in range(4)]); s.out=nn.Linear(n,H)
    def forward(s,x):
        B=x.shape[0]; h=torch.zeros(B,s.n); c=torch.zeros(B,s.n)
        for t in range(x.shape[1]):
            v=torch.atan(s.inp(torch.cat([x[:,t],h],1)))
            f,i,g,o=[m(v) for m in s.vqc]
            f=torch.sigmoid(f*3); i=torch.sigmoid(i*3); g=torch.tanh(g*2); o=torch.sigmoid(o*3)
            c=f*c+i*g; h=o*torch.tanh(c)
        return s.out(h)
class LSTM(nn.Module):
    def __init__(s,nf=6,hid=8,H=6):
        super().__init__(); s.l=nn.LSTM(nf,hid,batch_first=True); s.o=nn.Linear(hid,H)
    def forward(s,x): return s.o(s.l(x)[0][:,-1])
nparams=lambda m: sum(p.numel() for p in m.parameters())
print('QLSTM',nparams(QLSTM()),'| LSTM-4',nparams(LSTM(hid=4)),'| LSTM-8',nparams(LSTM(hid=8)))

## 3. Training and evaluation utilities

In [ ]:
def prep(nocal=False):
    X=Xw.copy()
    if nocal: X[:,:,3:6]=0                      # remove day-of-week, exam, holiday inputs
    return [torch.tensor(a) for a in (X[TR],Y[TR],X[VA],Y[VA],X[TE],Y[TE])]
def metrics(P,Yt):
    P=np.asarray(P)*100; Yt=np.asarray(Yt)*100; e=P-Yt; m=Yt>=5
    return dict(MAE=float(np.abs(e).mean()),RMSE=float(np.sqrt((e**2).mean())),MAPE=float((np.abs(e)[m]/Yt[m]).mean()*100),
                MAE_h=[float(v) for v in np.abs(e).mean(0)])
def exam_split(P):
    ex=Xw[TE][:,-1,4]==1; Yt=Y[TE]
    return dict(exam=metrics(P[ex],Yt[ex])['MAE'],nonexam=metrics(P[~ex],Yt[~ex])['MAE'])
def train(model,data,epochs,lr,bs,seed,tag=''):
    torch.manual_seed(seed); np.random.seed(seed)
    xtr,ytr,xva,yva,xte,yte=data; opt=torch.optim.Adam(model.parameters(),lr=lr)
    best=(1e9,None); hist=[]; t0=time.time()
    for ep in range(epochs):
        model.train(); perm=torch.randperm(len(xtr)); tl=0
        for i in range(0,len(xtr),bs):
            b=perm[i:i+bs]; opt.zero_grad(); loss=((model(xtr[b])-ytr[b])**2).mean(); loss.backward(); opt.step(); tl+=loss.item()*len(b)
        model.eval()
        with torch.no_grad(): vl=((model(xva)-yva)**2).mean().item()
        hist.append((tl/len(xtr),vl))
        if vl<best[0]: best=(vl,copy.deepcopy(model.state_dict()))
        if tag and (ep%5==0 or ep==epochs-1): print(f'{tag} epoch {ep+1:3d}  train {tl/len(xtr):.5f}  val {vl:.5f}  ({time.time()-t0:.0f}s)')
    model.load_state_dict(best[1]); model.eval()
    with torch.no_grad(): P=model(xte).numpy()
    return P,hist,time.time()-t0
R={}

## 4. Train the QLSTM (4 qubits, 2 VQC layers)

In [ ]:
R['QLSTM']={}
for s in CFG['seeds']:
    m=QLSTM(6,4,2); P,h,t=train(m,prep(),CFG['q_epochs'],0.02,128,s,tag=f'QLSTM seed{s}')
    R['QLSTM'][s]=dict(m=metrics(P,Y[TE]),hist=h,time=t,params=nparams(m),pred=P,exam=exam_split(P))
    print(f'  -> seed {s}: test MAE {R["QLSTM"][s]["m"]["MAE"]:.2f}  RMSE {R["QLSTM"][s]["m"]["RMSE"]:.2f}  params {nparams(m)}  time {t:.0f}s')
    if s==CFG['seeds'][0]: torch.save(m.state_dict(),'qlstm.pt')      # checkpoint used by the MVP app

## 5. Baselines: classical LSTM, naive, ARIMA

In [ ]:
for name,hid in (('LSTM4',4),('LSTM8',8)):
    R[name]={}
    for s in CFG['seeds']:
        m=LSTM(6,hid); P,h,t=train(m,prep(),CFG['l_epochs'],0.01,64,s)
        R[name][s]=dict(m=metrics(P,Y[TE]),hist=h,time=t,params=nparams(m),pred=P,exam=exam_split(P))
    print(name,'MAE per seed',[round(v['m']['MAE'],2) for v in R[name].values()])
R['LSTM8_nocal']={}
for s in CFG['seeds']:
    m=LSTM(6,8); P,h,t=train(m,prep(True),CFG['l_epochs'],0.01,64,s); R['LSTM8_nocal'][s]=dict(m=metrics(P,Y[TE]),exam=exam_split(P))
yt=df.occ.values/CAP; T=idx[TE]
naive={'Persistence':np.stack([[yt[i]]*H for i in T]),
       'Seasonal naive (daily)':np.stack([[yt[i+k-14] for k in range(1,H+1)] for i in T]),
       'Seasonal naive (weekly)':np.stack([[yt[i+k-98] for k in range(1,H+1)] for i in T])}
R['naive']={k:dict(m=metrics(v,Y[TE]),exam=exam_split(v)) for k,v in naive.items()}
from statsmodels.tsa.arima.model import ARIMA
ntr=int(0.7*(idx.max()+H+1)); fit=ARIMA(yt[:ntr],order=(3,0,1),trend='c').fit()
Pa=np.array([fit.apply(yt[:i+1],refit=False).forecast(H) for i in T])    # re-condition on observed data at each origin
R['ARIMA']=dict(m=metrics(Pa,Y[TE]),exam=exam_split(Pa))
print('baselines done')

## 6. Ablations: qubits, circuit depth, calendar features

In [ ]:
A={}
def abl(key,model,data,ep):
    P,h,t=train(model,data,ep,0.02,128,0,tag=key); A[key]=dict(m=metrics(P,Y[TE]),params=nparams(model),time=t,exam=exam_split(P)); print(f'  -> {key}: MAE {A[key]["m"]["MAE"]:.2f}')
abl('2 qubits, 2 layers',QLSTM(6,2,2),prep(),CFG['abl_epochs'])
abl('4 qubits, 1 layer', QLSTM(6,4,1),prep(),CFG['abl_epochs'])
abl('4 qubits, 2 layers',QLSTM(6,4,2),prep(),CFG['abl_epochs'])
abl('4 qubits, 3 layers',QLSTM(6,4,3),prep(),CFG['abl_epochs'])
abl('6 qubits, 2 layers',QLSTM(6,6,2),prep(),CFG['abl_epochs'])
abl('4q,2L, no calendar',QLSTM(6,4,2),prep(True),CFG['nocal_epochs'])

## 7. Results tables (these are the numbers in the report)

In [ ]:
mean=lambda d,k: np.mean([x['m'][k] for x in d.values()]); std=lambda d,k: np.std([x['m'][k] for x in d.values()])
exm=lambda d,k: np.mean([x['exam'][k] for x in d.values()])
rows=[]
for k,v in R['naive'].items(): rows.append((k,0,v['m']['MAE'],0,v['m']['RMSE'],v['m']['MAPE'],v['exam']['exam'],v['exam']['nonexam']))
a=R['ARIMA']; rows.append(('ARIMA(3,0,1)',6,a['m']['MAE'],0,a['m']['RMSE'],a['m']['MAPE'],a['exam']['exam'],a['exam']['nonexam']))
for nm,key in (('LSTM-4','LSTM4'),('LSTM-8','LSTM8'),('QLSTM (4 qubits)','QLSTM')):
    d=R[key]; rows.append((nm,d[CFG['seeds'][0]]['params'],mean(d,'MAE'),std(d,'MAE'),mean(d,'RMSE'),mean(d,'MAPE'),exm(d,'exam'),exm(d,'nonexam')))
T1=pd.DataFrame(rows,columns=['Model','Params','MAE','MAE_sd','RMSE','MAPE_%','MAE_exam','MAE_other']).round(2)
print('TABLE 4.1  Test-set accuracy (percentage points of capacity)'); print(T1.to_string(index=False))
print('\nMAE by horizon (hours ahead 1..6)')
for nm,key in (('QLSTM','QLSTM'),('LSTM-8','LSTM8'),('LSTM-4','LSTM4')): print(f'{nm:8s}',np.round(np.mean([x['m']['MAE_h'] for x in R[key].values()],0),2))
print('ARIMA   ',np.round(R['ARIMA']['m']['MAE_h'],2)); print('Seasonal weekly',np.round(R['naive']['Seasonal naive (weekly)']['m']['MAE_h'],2))
T2=pd.DataFrame([(k,v['params'],v['m']['MAE'],v['exam']['exam'],v['time']) for k,v in A.items()],columns=['Configuration','Params','MAE','MAE_exam','Sim_time_s']).round(2)
print('\nTABLE 4.2  Ablations (single seed)'); print(T2.to_string(index=False))
print('\nCalendar features: LSTM-8 with %.2f / without %.2f MAE'%(mean(R['LSTM8'],'MAE'),mean(R['LSTM8_nocal'],'MAE')))
print('Training time: QLSTM %.0fs vs LSTM-8 %.0fs per run'%(np.mean([x['time'] for x in R['QLSTM'].values()]),np.mean([x['time'] for x in R['LSTM8'].values()])))
open('results_summary.txt','w').write(T1.to_string(index=False)+'\n\n'+T2.to_string(index=False))

## 8. Figures

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(14,3.4))
names=[r[0] for r in rows]; ax[0].bar(range(len(rows)),[r[2] for r in rows],color=['#c3c2b7']*3+['#9a9993','#eb6834','#eb6834','#2a78d6'])
ax[0].set_xticks(range(len(rows))); ax[0].set_xticklabels([n.replace(' (','\n(') for n in names],fontsize=7,rotation=20); ax[0].set_title('Test MAE (% of capacity)')
hz=np.arange(1,7)
for nm,key,c in (('QLSTM','QLSTM','#2a78d6'),('LSTM-8','LSTM8','#eb6834'),('LSTM-4','LSTM4','#eb6834')): ax[1].plot(hz,np.mean([x['m']['MAE_h'] for x in R[key].values()],0),marker='o',color=c,ls='--' if nm=='LSTM-4' else '-',label=nm)
ax[1].plot(hz,R['ARIMA']['m']['MAE_h'],marker='o',color='#9a9993',label='ARIMA'); ax[1].legend(); ax[1].set_title('Error by horizon'); ax[1].set_xlabel('hours ahead')
s0=CFG['seeds'][0]; ax[2].plot(Y[TE][:,0]*100,'k',lw=1,label='Actual'); ax[2].plot(R['LSTM8'][s0]['pred'][:,0]*100,color='#eb6834',lw=1,label='LSTM-8'); ax[2].plot(R['QLSTM'][s0]['pred'][:,0]*100,color='#2a78d6',lw=1,label='QLSTM')
ax[2].legend(); ax[2].set_title('1-hour-ahead forecast, test set'); plt.tight_layout(); plt.show()